In [1]:
import pandas as pd
import numpy as np

# 1. Carregar a base original (Chácara)

In [2]:
# Carrega o arquivo de entrada (arquivo organizado em `data/filling_Ceps/`)
df = pd.read_csv('../../data/filling_Ceps/Elvira - Chacara - INEP 35107700 Dados ADS_coords_corrigidas_com_enderecos.csv')

# 2. Tratar a renda para número

In [3]:
import re

def parse_renda_seguro(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    s = re.sub(r"[^0-9,\.]", "", s)
    if "," in s:
        s = s.replace(".", "")
    elif s.count(".") > 1:
        s = s.replace(".", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


df["renda_media_num"] = df["renda_media"].apply(parse_renda_seguro)


# 3. Criar Total_0_9 (0–4 + 5–9 anos)

In [4]:
df["Total_0_9"] = (
    df["v01031_0_4anos"].fillna(0) +
    df["v01032_5_9anos"].fillna(0)
)

In [5]:
import math

def haversine_km(lat, lon, lat0, lon0):
    lat = np.radians(pd.to_numeric(lat, errors="coerce"))
    lon = np.radians(pd.to_numeric(lon, errors="coerce"))
    lat0 = math.radians(lat0)
    lon0 = math.radians(lon0)
    dlat = lat - lat0
    dlon = lon - lon0
    a = np.sin(dlat / 2) ** 2 + np.cos(lat0) * np.cos(lat) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

# Coordenadas da unidade
escola_lat = -23.6336779
escola_lon = -46.7132727

# Distancia por ponto
if "latitude_centro" in df.columns and "longitude_centro" in df.columns:
    df["distancia_km"] = haversine_km(df["latitude_centro"], df["longitude_centro"], escola_lat, escola_lon)
else:
    df["distancia_km"] = np.nan

# Parametros da filtragem por robustez e tamanho das listas
min_pontos = 5
top_n = 12
top_videos = 8


# 4. Corrigir renda para 2025 (inflação)

In [6]:
inflation_factor = 1.155
df["renda_atualizada_2025"] = df["renda_media_num"] * inflation_factor

# 5. Score linha a linha (informativo, não consolidado)

In [7]:
df["score_trafego_2025"] = df["renda_atualizada_2025"] * df["Total_0_9"]

# 6. Consolidar por CEP (SEM SOMAR — usar média!)

In [8]:
df_cep = (
    df.groupby("CEP", as_index=False)
      .agg({
          "Bairro": lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0],
          "renda_atualizada_2025": "median",
          "Total_0_9": "median",
          "populacao_total": "median",
          "distancia_km": "median",
      })
)

pontos = df.groupby("CEP").size().reset_index(name="pontos")
df_cep = df_cep.merge(pontos, on="CEP", how="left")


In [9]:
# Renomear colunas para refletir que sao medianas

df_cep = df_cep.rename(columns={
    "renda_atualizada_2025": "renda_mediana_2025",
    "Total_0_9": "mediana_criancas_0_9",
    "populacao_total": "populacao_mediana",
    "distancia_km": "distancia_mediana_km",
})


# 7. Score final no nível do CEP (correto)

In [10]:
df_cep["score_trafego_2025"] = (
    df_cep["renda_mediana_2025"] * df_cep["mediana_criancas_0_9"]
)

# 8. Ranking final

In [11]:
top_ceps = df_cep.sort_values("score_trafego_2025", ascending=False)

print(top_ceps.head(15))

           CEP                            Bairro  renda_mediana_2025  \
585  05635-050                Jardim Monte Kemel         24449.50200   
382  04719-905  Chácara Santo Antônio (Zona Sul)         32647.68045   
739  05709-040                       Vila Suzana         36272.94825   
413  04729-060                  Jardim Caravelas         19448.42130   
62   04564-900                    Cidade Monções         34184.38485   
253  04660-000                  Jardim Marajoara         29669.06250   
110  04583-909                     Vila Cordeiro         28934.10135   
397  04726-160                     Vila Cruzeiro         24868.25880   
689  05679-050           Jardim Panorama D'Oeste         35331.79650   
574  05634-001                Jardim Monte Kemel         22594.04070   
797  05726-140                      Vila Andrade         19572.22575   
415  04730-000                   Várzea de Baixo         18074.72205   
71   04566-905                    Cidade Monções         34482.8

In [12]:
# Arredondar para 2 casas decimais (padrao monetario)
top_ceps["renda_mediana_2025"] = top_ceps["renda_mediana_2025"].round(2)
top_ceps["score_trafego_2025"] = top_ceps["score_trafego_2025"].round(2)
if "distancia_mediana_km" in top_ceps.columns:
    top_ceps["distancia_mediana_km"] = top_ceps["distancia_mediana_km"].round(2)


In [13]:
# Salvar resultado agregado na pasta do notebook (comportamento original)
top_ceps.to_csv("chacara_top_ceps_2025.csv", index=False)

In [14]:
# Lista dos bairros desejados
bairros_desejados = ['Granja Julieta', 'Ch?cara Santo Ant?nio', 'Santo Amaro', 'Ch?cara Flora', 'Jardim Santo Amaro', 'Vila Andrade']

# Filtrar diretamente no top_ceps
top_ceps_filtrado = top_ceps[top_ceps["Bairro"].isin(bairros_desejados)]
top_ceps_filtrado = top_ceps_filtrado[top_ceps_filtrado["pontos"] >= min_pontos]

top_ceps_filtrado


,CEP,Bairro,renda_mediana_2025,mediana_criancas_0_9,populacao_mediana,distancia_mediana_km,pontos,score_trafego_2025
796,05726-130,Vila Andrade,11951.80,92.0,600.0,3.06,5,1099565.73
823,05734-080,Vila Andrade,9578.44,56.0,390.0,3.48,5,536392.53
778,05717-270,Vila Andrade,15458.43,31.0,357.0,2.29,5,479211.26
803,05727-240,Vila Andrade,12706.02,34.0,237.0,2.75,5,432004.56
829,05734-150,Vila Andrade,13601.75,25.5,297.0,2.65,6,346844.57
810,05729-090,Vila Andrade,10568.17,29.0,265.0,2.56,5,306476.91
791,05724-902,Vila Andrade,9531.41,32.0,377.0,2.23,5,305005.01
808,05728-050,Vila Andrade,3787.58,61.5,414.5,3.07,6,232936.17
809,05729-080,Vila Andrade,1905.29,93.5,602.5,2.80,6,178144.97


In [15]:
top_ceps_filtrado.to_csv("chacara_top_ceps_filtrados_2025.csv", index=False)

In [16]:
# Rankings por distancia + score (logica de videos)
top_ceps_base = top_ceps[top_ceps["pontos"] >= min_pontos].copy()

top_ceps_proximos = top_ceps_base.sort_values("distancia_mediana_km").head(top_n)
top_ceps_videos = top_ceps_proximos.sort_values("score_trafego_2025", ascending=False).head(top_videos)

top_ceps_proximos.to_csv("chacara_top_ceps_proximos_2025.csv", index=False)
top_ceps_videos.to_csv("chacara_top_ceps_videos_2025.csv", index=False)

top_ceps_filtrado_base = top_ceps_filtrado.copy()
top_ceps_filtrado_proximos = top_ceps_filtrado_base.sort_values("distancia_mediana_km").head(top_n)
top_ceps_filtrado_videos = top_ceps_filtrado_proximos.sort_values("score_trafego_2025", ascending=False).head(top_videos)

top_ceps_filtrado_proximos.to_csv("chacara_top_ceps_filtrados_proximos_2025.csv", index=False)
top_ceps_filtrado_videos.to_csv("chacara_top_ceps_filtrados_videos_2025.csv", index=False)
